# 02 | Leis registradas pela RAND

Transformamos regras legais em colunas com 0 ou 1. A regra atual usa o **ano** de vigência; portanto, não mede quantos meses uma lei esteve em vigor.

**Como usar:** execute os blocos de código 1, 2 e 3, nessa ordem. Leia a explicação antes de cada bloco. Cada bloco reúne pequenas ações que fazem parte da mesma etapa.

## Bloco 1: Abrir e conhecer

A planilha oficial RAND já está na pasta `reports/attachments/`. Porte de arma escondida é chamado `CCW` na planilha.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
import pandas as pd
arquivo = ROOT / "reports/attachments/TL-A243-2-v4 State Firearm Law Database 6.0.xlsx"
leis = pd.read_excel(arquivo, sheet_name="Database")
porte = leis[leis["Law Class"].eq("carrying a concealed weapon (ccw)")].copy()
privadas = leis[(leis["Law Class"] == "background checks") & (leis["Law Class Subtype"] == "private sales : point of sale")].copy()
print("Registros de CCW:", len(porte), "| Antecedentes em vendas privadas:", len(privadas))

## Bloco 2: Criar as colunas

Para cada estado e ano, consultamos as categorias da RAND ainda não substituídas. `may issue`: decisão da autoridade; `shall issue`: licença se cumpre requisitos; `permitless`: sem exigência de licença nessa modalidade.

In [ ]:
registros_ccw = []
for estado in sorted(porte["State"].unique()):
    linhas = porte[porte["State"] == estado]
    for ano in range(2019, 2025):
        vigentes = linhas[(linhas["Effective Date Year"] <= ano) & (linhas["Supersession Date Year"].isna() | (linhas["Supersession Date Year"] > ano))]
        tipos = set(vigentes["Law Class Subtype"])
        registros_ccw.append({"state": estado, "year": ano,
            "ccw_shall_issue": int("shall issue" in tipos),
            "ccw_permitless": int("shall issue (permit not required)" in tipos),
            "ccw_may_issue": int("may issue" in tipos),
            "ccw_prohibited": int("prohibited" in tipos)})
ccw = pd.DataFrame(registros_ccw).query("state != 'District of Columbia'").copy()
registros_bg = []
for estado in sorted(ccw["state"].unique()):
    linhas = privadas[privadas["State"] == estado]
    for ano in range(2019, 2025):
        vigentes = linhas[(linhas["Effective Date Year"] <= ano) & (linhas["Supersession Date Year"].isna() | (linhas["Supersession Date Year"] > ano)) & (linhas["Type of Change"] != "Repeal")]
        tipos = set(vigentes["Handguns or Long Guns"])
        registros_bg.append({"state": estado, "year": ano,
            "background_handgun": int(bool(tipos & {"handgun", "handgun and long gun"})),
            "background_longgun": int(bool(tipos & {"long gun", "handgun and long gun"}))})
background = pd.DataFrame(registros_bg)
print("CCW:", ccw.shape, "| Antecedentes:", background.shape)

## Bloco 3: Conferir e salvar

Exigimos 300 registros sem repetições. **Importante:** modalidades podem coexistir; isso não significa necessariamente um erro.

In [ ]:
for nome, tabela in [("CCW", ccw), ("Background", background)]:
    assert tabela.shape[0] == 300, f"{nome}: linhas inesperadas"
    assert tabela["state"].nunique() == 50
    assert not tabela.duplicated(["state","year"]).any()
    destino = ROOT / ("data/processed/rand_ccw_2019_2024.csv" if nome == "CCW" else "data/processed/rand_background_2019_2024.csv")
    destino.parent.mkdir(parents=True, exist_ok=True)
    tabela.to_csv(destino, index=False)
    print(nome, "salvo:", destino.relative_to(ROOT))
print("Estados-anos com mais de uma modalidade:", int((ccw[["ccw_shall_issue","ccw_permitless","ccw_may_issue","ccw_prohibited"]].sum(axis=1)>1).sum()))